In [ ]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import trange

DEVICE = "cuda"
DEVICE = "mps"

DATA = open("shakespeare.txt", "r").read()
print(len(DATA))

VOCAB = list(sorted(set(DATA)))
VOCAB_MAP = {v:k for k,v in enumerate(VOCAB)}
VOCAB_SIZE = len(VOCAB)

def encode(s):
    return torch.tensor([VOCAB_MAP[x] for x in s], device = DEVICE)

def decode(l):
    return ''.join(VOCAB[x] for x in l)

DATA_TRAIN = encode(DATA[:int(0.9*len(DATA))])
DATA_TEST = encode(DATA[int(0.9*len(DATA)):])

'''
Alot of time spent in generate() as my logic didn't properly handle variable T length, fixed eventually
Otherwise seems fast and precise.

No Bugs

To Think About:
  - clipping
  - warmup
  - lr scheduler
  - init


'''

class MultiHeadAttention(nn.Module):
    def __init__(self, context_window, embedding_dim, head_size, dropout_p):
        super().__init__()
        assert embedding_dim % head_size == 0, "Headsize need to be divisor of embedding dimensions"
        self.head_size = head_size
        self.qkv = nn.Linear(embedding_dim, embedding_dim * 3, bias=False)
        self.proj = nn.Linear(embedding_dim, embedding_dim)
        self.dropout = nn.Dropout(dropout_p)
        #self.register_buffer("causal_mask", torch.tril(torch.ones(context_window, context_window, dtype=torch.bool)))
        ang = torch.arange(0,context_window).view(-1,1) * 1000.0 **( -2 * torch.arange(0,head_size//2) / head_size)
        self.register_buffer("rope_cos", torch.cos(ang), persistent=False)
        self.register_buffer("rope_sin", torch.sin(ang), persistent=False)
        
    def rope(self, x):
        B,H,T,W = x.shape
        xx, yy = x.chunk(2, dim=-1)
        cos = self.rope_cos[:T,]
        sin = self.rope_sin[:T,]
        return torch.cat([
            xx * cos - yy * sin, 
            xx * sin + yy * cos], 
            dim=-1)

    def forward(self, x):
        B,T,E = x.shape
        W = self.head_size
        H = E // W
        q,k,v = self.qkv(x).view(B,T,3,H,W).permute(2,0,3,1,4) # B,H,T,W
        q = self.rope(q)
        k = self.rope(k)
        a = F.scaled_dot_product_attention(q,k,v, is_causal=True, dropout_p=self.dropout.p if self.training else 0)
        #a = (q @ k.transpose(-2,-1)) / W**0.5  # B,H,T,T
        #a = a.masked_fill(~self.causal_mask[:T,:T], float('-inf'))
        #a = F.softmax(a, -1)
        #a = self.dropout(a)
        #a = a @ v # B,H,T,W
        return self.proj(a.transpose(-3,-2).reshape(B,T,E))

class FeedForward(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.lin = nn.Linear(embedding_dim, embedding_dim*4)
        self.gelu = nn.GELU()
        self.proj = nn.Linear(embedding_dim*4, embedding_dim)

    def forward(self, x):
        x = self.lin(x)
        x = self.gelu(x)
        x = self.proj(x)
        return x

class AttentionBlock(nn.Module):
    def __init__(self, context_window, embedding_dim, head_size, dropout_p):
        super().__init__()
        self.mha = MultiHeadAttention(
            context_window=context_window,
            embedding_dim=embedding_dim,
            head_size=head_size,
            dropout_p=dropout_p)
        self.ff = FeedForward(embedding_dim=embedding_dim)
        self.layn1 = nn.LayerNorm(embedding_dim)
        self.layn2 = nn.LayerNorm(embedding_dim)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, x):
        x = x + self.dropout(self.mha(self.layn1(x)))
        x = x + self.dropout(self.ff(self.layn2(x)))
        return x

class Transformer(nn.Module):
    def __init__(self, vocab_size, context_window, embedding_dim, attention_blocks, head_size, dropout_p):
        super().__init__()
        self.context_window = context_window
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        #self.embeddings_pos = nn.Embedding(context_window, embedding_dim)
        self.attention = nn.Sequential(*[
            AttentionBlock(
                context_window=context_window,
                embedding_dim=embedding_dim,
                head_size=head_size,
                dropout_p=dropout_p
            ) for _ in range(attention_blocks)
        ])
        self.dropout = nn.Dropout(dropout_p)
        self.layn = nn.LayerNorm(embedding_dim)
        self.lha = nn.Linear(embedding_dim, vocab_size, bias=False)
        self.embeddings.weight = self.lha.weight
        #self.register_buffer("pos_offsets", torch.arange(0, context_window))

    def forward(self, x, y=None):
        B,T = x.shape
        x = self.embeddings(x)# + self.embeddings_pos(self.pos_offsets[:T])
        x = self.dropout(x)
        x = self.attention(x)
        x = self.layn(x)
        x = self.lha(x)
        if y is None:
            return x
        else:
            return F.cross_entropy(x.view(B*T,-1), y.reshape(B*T))

    @torch.no_grad()
    def generate(self, num, prompt=""):
        out = [0] if len(prompt)==0 else [VOCAB_MAP[x] for x in prompt]
        was_train = self.training
        self.eval()
        for _ in range(num):
            i = torch.tensor(out[-self.context_window:], device = self.lha.weight.device).view(1,-1)
            l = self(i)
            x = torch.multinomial(F.softmax(l[0,-1], dim=-1),1).item()
            out.append(x)
        self.train(was_train)
        return decode(out[1 if len(prompt)==0 else 0:])

def get_batch(data, context_window, batch_size):
    idx = torch.randint(len(data)-context_window, (batch_size, 1), device = data.device)
    idx = idx + torch.arange(0, context_window+1, device = data.device)
    return data[idx[:,:-1]], data[idx[:,1:]]

@torch.no_grad()
def estimate_loss(model):
    was_train = model.training
    model.eval()
    res =  [
        model(*get_batch(data, model.context_window, 100))
        for data in [DATA_TEST, DATA_TRAIN]]
    model.train(was_train)
    return res

m = Transformer(
    VOCAB_SIZE, 
    context_window=64, 
    embedding_dim=128, 
    head_size=32, 
    attention_blocks=8, 
    dropout_p=0.1
).to(DATA_TRAIN.device)
print(f"Model Params: {sum(x.numel() for x in m.parameters()):,}")
o = optim.AdamW(m.parameters(), 1e-3)

print(estimate_loss(m))

def train_loop(model, opt, batch_size, iterations):
    pbar = trange(iterations, desc="Training")
    for i in pbar:
        opt.zero_grad()
        loss = model(*get_batch(DATA_TRAIN, model.context_window, batch_size))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if i%100==99:
            l_ts, l_tr = estimate_loss(m)
            pbar.set_postfix({
                "test_loss": f"{l_ts:.4}",
                "train_loss": f"{l_tr:.4}"
            })

train_loop(m,o,256,1000)
print(estimate_loss(m))
print(m.generate(1000))
print("Done!")

# [pure] 1.6918 @ 01:09
# [fuse] 1.6632 @ 00:54
# [rope] 1.5160 @ 01:03



1115394
Model Params: 1,591,680
[tensor(4.3515, device='cuda:0'), tensor(4.3643, device='cuda:0')]


Training: 100%|██████████| 1000/1000 [01:03<00:00, 15.80it/s, test_loss=1.574, train_loss=1.372]


[tensor(1.5508, device='cuda:0'), tensor(1.3550, device='cuda:0')]
I'll present not: my lord, and king like swirt
There can bear to be from tenderous?
Have aside emploach'd their own blood,
Be not take in English'd, he burning with my
names conquests gone! I
BUSHY:
Go provide Angelo mistress of your tack-blind,
Like a love make pursuing soily! Where so much
Woman journey.
'Tis he dream the death night some like what
is what I not. Till here it, manners,
And all to my hould surperitable. Where she's ded!

GLOUCESTER:
Suffer the stand red night? be to my beoved
Here's the tughtly soldiers not what finger to thine eyes
Be not a mine met on my senator: and proclaim
My Romeo souling. You tribunes I know,
Which to Duke man instant's labour open
At thy matter this appear, Warwick, and holy tigely
When he been their brothers of lawful horse:
Well, hear me you for preety as things in my other
Tent shortest thousand, that fair less'. What cannot be.

DUCHESS OF YORK:
Very blush and kill'd them.
